# Problem Set: Neural Networks

<span style="color:red">Warning!</span> Some problems here require training neural
networks, which can take a few minutes each - start early so you don't get stuck.

<span style="color:red">Note:</span> The intro-to-PyTorch lab and the Day 1 slides are
your friends - many of the examples there map directly onto these questions.

## Cars vs. Trucks: Feedforward Neural Networks

Your goal is to train **feedforward (fully-connected) neural networks** to classify
whether the vehicle in an image is a car (automobile) or a truck. We use only the
tools from lecture: tensors, `nn.Module`, activation functions, cross-entropy loss,
optimizers (SGD / Adam), the training loop, and regularization (dropout / weight
decay). No convolutions.

Use PyTorch. It may help to prototype on a small subset of the images first.

## Part I - Data

### Question 1: Load data + exploratory analysis

We'll use the [CIFAR-10](https://en.wikipedia.org/wiki/CIFAR-10) dataset, a classic
benchmark in ML. It has 10 classes; we'll use just two of them for a binary task.

Helper code to load the data is provided below.

**Your tasks:**
- Create a subset of CIFAR-10 keeping only the **automobile** and **truck** classes.
- Select 9 random images from your training subset and plot them in a 3 x 3 grid,
  each titled with its label.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

def loadCifar10(data_path):
    "load CIFAR-10 train and test datasets (pixels scaled to [0, 1])"
    transform = transforms.Compose([transforms.ToTensor()])
    cifar_train = datasets.CIFAR10(data_path, train=True, download=True, transform=transform)
    cifar_test = datasets.CIFAR10(data_path, train=False, download=True, transform=transform)
    return cifar_train, cifar_test

In [ ]:
cifar_train, cifar_test = loadCifar10("./data/")
print(cifar_train.classes)

In [ ]:
# automobile is class 1, truck is class 9; relabel to 0 and 1
keep = {1: 0, 9: 1}
label_name = {0: "automobile", 1: "truck"}

train_idx = [i for i, t in enumerate(cifar_train.targets) if t in keep]
test_idx = [i for i, t in enumerate(cifar_test.targets) if t in keep]
train_sub = Subset(cifar_train, train_idx)
test_sub = Subset(cifar_test, test_idx)
print(f"train subset: {len(train_sub)} images, test subset: {len(test_sub)} images")

fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for ax in axes.ravel():
    img, target = train_sub[np.random.randint(len(train_sub))]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(label_name[keep[target]])
    ax.axis("off")
plt.tight_layout()
plt.show()

### Question 2: Prepare the data for a feedforward network

A feedforward network takes a flat **vector**, not a 3 x 32 x 32 image. Get the data
ready:

- Build flattened feature tensors `x_train`, `x_test` (each image becomes a length-3072
  vector) and integer label tensors `y_train`, `y_test` (0 = automobile, 1 = truck).
- Wrap them in `DataLoader`s with batch size 32 (shuffle the training loader).

**Short answer:** how many input features does each image become, and why must we
flatten the image before feeding it to a fully-connected network?

In [ ]:
def toTensors(subset):
    "stack a cifar subset into a flat feature matrix and a 0/1 label vector"
    xs, ys = [], []
    for img, target in subset:
        xs.append(img.reshape(-1))          # flatten 3 x 32 x 32 -> 3072
        ys.append(keep[target])
    return torch.stack(xs), torch.tensor(ys, dtype=torch.long)

x_train, y_train = toTensors(train_sub)
x_test, y_test = toTensors(test_sub)
print("x_train shape:", x_train.shape)      # (N, 3072)

batch_size = 32
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)

**Answer:** each image becomes **3 x 32 x 32 = 3072** features. A fully-connected
(`nn.Linear`) layer multiplies the input by a weight matrix, so it expects a 1-D
vector per example; flattening turns the image grid into that vector. (The cost is
that we throw away the spatial layout of the pixels - which is exactly what
convolutional networks later keep.)

## Part II - Neural Networks

### Question 3: Helper functions

Complete two helpers (the intro-to-PyTorch lab is a good reference):

- `trainModel`: runs the training loop and returns the list of average epoch losses.
- `validate`: given a model and a data loader, returns the **accuracy, precision,
  recall, and F1-score**.

**Note:** code the metrics **manually** (counts of true/false positives and
negatives) - do not use scikit-learn or any other library for them. Treat
truck (label 1) as the positive class.

In [ ]:
def trainModel(model, n_epochs, optimizer, loss_fn, data_loader):
    "train a model; return the list of average epoch losses"
    history = []
    for epoch in range(n_epochs):
        model.train()
        running = 0.0
        for xb, yb in data_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
        avg = running / len(data_loader.dataset)
        history.append(avg)
        if (epoch + 1) % 10 == 0:
            print(f"epoch {epoch + 1}/{n_epochs}  loss {avg:.4f}")
    return history

def validate(model, data_loader):
    "accuracy, precision, recall, f1 for the positive class (truck = 1), computed by hand"
    model.eval()
    tp = fp = tn = fn = 0
    with torch.no_grad():
        for xb, yb in data_loader:
            preds = model(xb).argmax(dim=1)
            tp += int(((preds == 1) & (yb == 1)).sum())
            fp += int(((preds == 1) & (yb == 0)).sum())
            tn += int(((preds == 0) & (yb == 0)).sum())
            fn += int(((preds == 0) & (yb == 1)).sum())
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

### Question 4: A single-hidden-layer feedforward network

Build a feedforward network and train it.

**Architecture:**
- Input: 3072 features
- Hidden layer: 256 nodes, ReLU activation
- Output: 2 logits (one per class)

**Compile / train:**
- Loss: cross-entropy
- Optimizer: Adam
- Batch size: 32
- At least 100 epochs

Plot the average epoch loss, then report accuracy, precision, recall, and F1 on
**both** the training and test sets.

In [ ]:
class FFN(nn.Module):
    "single hidden layer: 3072 -> 256 (relu) -> 2 logits"
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3 * 32 * 32, 256),
            nn.ReLU(),
            nn.Linear(256, 2),
        )

    def forward(self, x):
        return self.net(x)

ffn = FFN()
optimizer = optim.Adam(ffn.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
history = trainModel(ffn, 100, optimizer, loss_fn, train_loader)

plt.plot(history)
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.title("FFN training loss")
plt.show()

print("train:", validate(ffn, train_loader))
print("test: ", validate(ffn, test_loader))

### Question 5: Your turn - build a better network

Design your own feedforward network, `NewNet`, aiming to **exceed an F1-score of
0.80** on the test set. You don't need formal cross-validation or hyperparameter
search - just experiment locally until you have an architecture that works.

Knobs you can turn (all from lecture):
- number and width of hidden layers
- activation functions (ReLU, tanh, sigmoid)
- dropout
- learning rate
- optimizer (SGD vs Adam) and weight decay
- number of epochs

(Feedforward only - no convolutions.) Train for at least 100 epochs, then report the
metrics on the train and test sets.

In [ ]:
class NewNet(nn.Module):
    "deeper mlp with dropout"
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3 * 32 * 32, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.net(x)

net = NewNet()
optimizer = optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
history = trainModel(net, 100, optimizer, loss_fn, train_loader)

plt.plot(history)
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.title("NewNet training loss")
plt.show()

print("train:", validate(net, train_loader))
print("test: ", validate(net, test_loader))

**Note (instructor):** on raw flattened CIFAR pixels a fully-connected network
typically tops out around an F1 of ~0.80-0.85 for car-vs-truck. That ceiling is the
point: MLPs ignore spatial structure, which is exactly what motivates convolutional
networks later in the course. A deeper net with dropout + a little weight decay,
trained ~100 epochs, should clear 0.80.

### Question 6: Overfitting and regularization

Show overfitting, then fix it.

- Train a **large** feedforward network (e.g. two wide hidden layers, no dropout) and
  record **both** train and test accuracy across epochs. You should see the train
  accuracy climb while the test accuracy stalls or drops - the train/test gap is
  overfitting.
- Now train a network of the **same capacity** but with a regularizer from lecture
  (dropout and/or weight decay). Show that the train/test gap shrinks.
- **Short answer:** what is overfitting, and why does your chosen fix help?

In [ ]:
def trainTrackingAccuracy(model, optimizer, loss_fn, n_epochs=40):
    "train, recording train and test accuracy after each epoch"
    train_acc, test_acc = [], []
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()
        train_acc.append(validate(model, train_loader)["accuracy"])
        test_acc.append(validate(model, test_loader)["accuracy"])
    return train_acc, test_acc

loss_fn = nn.CrossEntropyLoss()

# big, unregularized -> overfits
big = nn.Sequential(nn.Linear(3072, 1024), nn.ReLU(),
                    nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 2))
tr_big, te_big = trainTrackingAccuracy(big, optim.Adam(big.parameters(), lr=1e-3), loss_fn)

# same capacity + dropout + weight decay -> less overfitting
reg = nn.Sequential(nn.Linear(3072, 1024), nn.ReLU(), nn.Dropout(0.5),
                    nn.Linear(1024, 1024), nn.ReLU(), nn.Dropout(0.5), nn.Linear(1024, 2))
tr_reg, te_reg = trainTrackingAccuracy(reg, optim.Adam(reg.parameters(), lr=1e-3, weight_decay=1e-3), loss_fn)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(tr_big, label="train"); axes[0].plot(te_big, label="test")
axes[0].set_title("no regularization (overfits)")
axes[1].plot(tr_reg, label="train"); axes[1].plot(te_reg, label="test")
axes[1].set_title("dropout + weight decay")
for a in axes:
    a.set_xlabel("epoch"); a.set_ylabel("accuracy"); a.legend(); a.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Answer:** overfitting is when the model memorizes the training data (train
accuracy keeps rising) but fails to generalize (test accuracy stalls or falls).
Dropout randomly zeroes hidden units during training so the network can't rely on
any single unit, and weight decay penalizes large weights - both push the model
toward simpler, smoother functions that generalize better, shrinking the
train/test gap.

### Extra credit (optional): SGD vs Adam

Train the single-hidden-layer `FFN` twice - once with `SGD` and once with `Adam`
(same learning rate) - and plot the two training-loss curves on the same axes.
Which converges faster here?

In [ ]:
loss_fn = nn.CrossEntropyLoss()

ffn_sgd = FFN()
h_sgd = trainModel(ffn_sgd, 60, optim.SGD(ffn_sgd.parameters(), lr=1e-3), loss_fn, train_loader)

ffn_adam = FFN()
h_adam = trainModel(ffn_adam, 60, optim.Adam(ffn_adam.parameters(), lr=1e-3), loss_fn, train_loader)

plt.plot(h_sgd, label="SGD")
plt.plot(h_adam, label="Adam")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("SGD vs Adam"); plt.legend()
plt.show()